In [2]:
import json
import os
import numpy as np
import pandas as pd
from datasets import load_dataset
from urllib.parse import parse_qs, urlparse
import requests
import chromadb
import openai
from chromadb.config import Settings
import os
import openai
from openai import OpenAI
import chromadb
import fitz  # PyMuPDF
from chromadb.utils import embedding_functions
import chromadb.utils.embedding_functions as embedding_functions
from chromadb.config import Settings

c:\Users\spurt\.conda\envs\Rag_research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\spurt\.conda\envs\Rag_research\Lib\site-packages\onnxruntime\capi\onnxruntime_validation.py:26: UserWarning: Unsupported Windows version (11). ONNX Runtime supports Windows 10 and above, only.
  warnings.warn(


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.vectorstores import Chroma
import base64
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os
from langchain.evaluation.qa import QAEvalChain
from bert_score import score
from openai import OpenAI

In [7]:
import fitz

In [6]:
def extract_pdf_url(url):
    """
    Extracts the actual PDF URL from the given URL.
    Decodes it from base64 if necessary.
    """
    if url.lower().endswith('.pdf'):
        return url  # Direct PDF URL
    else:
        parsed_url = urlparse(url)
        query_params = parse_qs(parsed_url.query)
        pdf_target = query_params.get('pdfTarget', [None])[0]

        if pdf_target:
            pdf_url = base64.b64decode(pdf_target).decode('utf-8')
            return pdf_url
        else:
            raise ValueError("No valid PDF URL found in the provided URL")

def download_pdf(url, save_path):
    """
    Downloads a PDF from a given URL.
    """
    try:
        pdf_url = extract_pdf_url(url)
        response = requests.get(pdf_url, stream=True)
        response.raise_for_status()  # Ensure the request was successful

        with open(save_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)

        print(f"Downloaded PDF from: {pdf_url} to {save_path}")
    except Exception as e:
        print(f"Error downloading PDF: {e}")

In [7]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        text += page.get_text()
    return text

In [6]:

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
                api_key= openai_api_key ,
                model_name="text-embedding-ada-002"
            )

In [8]:
def create_chroma_vectordb_from_pdf(pdf_path, openai_api_key, batch_size=100):
# Extract text from PDF
    text = extract_text_from_pdf(pdf_path)
    
    # Split text into sentences
    sentences = text.split('\n')
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]  # Remove empty sentences
    
    # Initialize OpenAI Embedding Function
    openai_ef = embedding_functions.OpenAIEmbeddingFunction(
        api_key=openai_api_key,
        model_name="text-embedding-ada-002"
    )
    
    # Batch processing for embeddings
    vectors = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        if len(batch) > 0:  # Ensure batch is not empty
            batch_vectors = openai_ef(batch)
            vectors.extend(batch_vectors)
    
    # Store vectors in Chroma vector database
    client = chromadb.Client(Settings())
    collection_name = "Finance_bench_documents"
    collection = client.get_or_create_collection(name= collection_name) 
    # if client.has_collection(collection_name):
    #     collection = client.get_collection(collection_name)
    # else:
    #     collection = client.create_collection(collection_name)
    
    for i, (sentence, vector) in enumerate(zip(sentences, vectors)):
        collection.add(f"id_{i}", vector, {"sentence": sentence})
    
    print(f"Stored {len(sentences)} vectors in the Chroma vector database.")

In [9]:
dataset = load_dataset("PatronusAI/financebench")
df = pd.DataFrame(dataset['train'])

In [12]:
df.columns

Index(['financebench_id', 'doc_name', 'doc_link', 'doc_period',
       'question_type', 'question', 'answer', 'evidence_text', 'page_number'],
      dtype='object')

In [10]:
test = df[:5]
test

,financebench_id,doc_name,doc_link,doc_period,question_type,question,answer,evidence_text,page_number
0,financebench_id_03029,3M_2018_10K,https://investors.3m.com/financials/sec-filing...,2018,metrics-generated,What is the FY2018 capital expenditure amount ...,$1577.00,Table of Contents \n3M Company and Subsidiarie...,60
1,financebench_id_04672,3M_2018_10K,https://investors.3m.com/financials/sec-filing...,2018,metrics-generated,Assume that you are a public equities analyst....,$8.70,Table of Contents \n3M Company and Subsidiarie...,58
2,financebench_id_00499,3M_2022_10K,https://investors.3m.com/financials/sec-filing...,2022,domain-relevant,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...",3M Company and Subsidiaries\n Consolidated Sta...,"48,50,52"
3,financebench_id_01226,3M_2022_10K,https://investors.3m.com/financials/sec-filing...,2022,domain-relevant,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,"SG&A, measured as a percent of sales, increase...",27
4,financebench_id_01865,3M_2022_10K,https://investors.3m.com/financials/sec-filing...,2022,novel-generated,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,Worldwide Sales Change\nBy Business Segment Or...,25


In [12]:
def query_openai_with_context(query, collection_name="Finance_bench_documents", top_k=2):
    # Initialize Chroma client
    client = chromadb.Client()
    
    collection = client.get_collection(collection_name)
    
    # Initialize OpenAI Embedding Function
    openai_ef = embedding_functions.OpenAIEmbeddingFunction(api_key=openai_api_key)
    
    # Generate the query vector
    query_vector = openai_ef(query)
    
    # Retrieve the most relevant vectors from Chroma
    #results = collection.query(vector=query_vector, top_k=top_k)
    results = collection.query(query_embeddings=query_vector, n_results=top_k)
    print(results)

    # Extract the context sentences
    # context_sentences = [result['sentence'] for result in results]
    # context = "\n".join(context_sentences)
    
    # Formulate the prompt for OpenAI with context
    template = """You are a financial chatbot trained to answer questions based on the information provided in 10-K
    documents. Your responses should be directly sourced from the content of these documents. When asked
    a question, ensure that your answer is explicitly supported by the text in the 10-K filing, and do not
    include any external information, interpretations, or assumptions not clearly stated in the document. If
    a question pertains to financial data or analysis that is not explicitly covered in the 10-K filing provided,
    respond by stating that the information is not available in the document. Your primary focus should
    be on accuracy, specificity, and adherence to the information in 10-K documents, particularly regarding
    financial statements, company performance, and market position."""
    
    prompt = f"\nContext:\n{results}\n\nQuery: {query}\n\nAnswer:"
    
    # Query the OpenAI model
    #openai.api_key = openai_api_key
    # client = OpenAI(api_key = openai_api_key)
    # response = client.chat.completions.create(
    #     model="gpt-3.5-turbo",  # Choose the appropriate model
    #     messages=[
    #         {"role": "system", "content": template},
    #         {"role": "user", "content": f"Context:\n{results}\n\nQuery: {query}\n\nAnswer:"}
    #     ],
    #     max_tokens=150
    # )

    messages = [
            {"role": "system", "content": template},
            {"role": "user", "content": prompt}
        ]
    response = get_assistant_response(messages)
    
    return response

In [13]:
def display_chat_history(messages):
    for message in messages:
        print(f"{message['role'].capitalize()}: {message['content']}")

def get_assistant_response(messages):
  client = OpenAI(api_key = openai_api_key)
  response = client.chat.completions.create(
      model="gpt-3.5-turbo",
      messages=[{"role": m["role"], "content": m["content"]} for m in messages],
  )

  return response.choices[0].message.content

In [19]:
# from sklearn.metrics.pairwise import cosine_similarity
# import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(text1, text2):
    # Create a TfidfVectorizer
    vectorizer = TfidfVectorizer()

    # Fit and transform the two texts
    tfidf_matrix = vectorizer.fit_transform([text1, text2])

    # Calculate cosine similarity
    cosine_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])

    # Extract the similarity score
    similarity_score = cosine_sim[0][0]
    
    return similarity_score


def calculate_bertscore(candidate, reference):
    P, R, F1 = score([candidate], [reference], lang="en", verbose=True)
    return P.mean().item()

In [15]:

def evaluate_llm_responses(question, model_answer, refrence_answer):
    for question in question:
        messages = [
            {"role": "system", "content": "You are an assistant that provides concise and accurate answers."},
            {"role": "user", "content": question}
        ]
        response = model_answer

    evaluation_scores = []
    for i in range(len(response)):
        evaluation_prompt = f"""
        Evaluate the following response against the reference answer. Assign a score between 0 and 1 based on correctness and provide a brief justification.

        Question: {question}
        Response: {model_answer}
        Reference Answer: {refrence_answer}

        Score (0 to 1):
        Justification:
        """
        messages = [
            {"role": "system", "content": "You are an evaluator that scores responses based on correctness."},
            {"role": "user", "content": evaluation_prompt}
        ]
        evaluation_response = get_assistant_response(messages)

        evaluation_text = evaluation_response.strip()
        try:
            score_line = evaluation_text.split('\n')[0]
            score = float(score_line.split(':')[1].strip())
            evaluation_scores.append(score)
        except Exception as e:
            print(f"Error parsing score: {e}")
            evaluation_scores.append(0.0)

    average_score = sum(evaluation_scores) / len(evaluation_scores) if evaluation_scores else 0
    print(f'Average Correctness Score: {average_score:.2f}')
    return average_score

Evaluation Code


In [22]:
# Initialize an empty list to collect data
results_list = []

for index, row in test.iterrows():
    download_dir = "pdf_documents"
    os.makedirs(download_dir, exist_ok=True)
    doc_link = row['doc_link']
    doc_name = row['doc_name']
    question = row['question']
    ref_answer = row['answer']
    ref_context = row['evidence_text']
    doc_path = os.path.join(download_dir, f"{doc_name}.pdf")

    download_pdf(doc_link, doc_path)
    create_chroma_vectordb_from_pdf(doc_path, openai_api_key)
    print("Querying Model now")
    model_answer = query_openai_with_context(question)
    print(model_answer)

    # Evaluation for structured QA 
    cosine_similarity_score = calculate_cosine_similarity(model_answer, ref_answer)
    bert_score = calculate_bertscore(model_answer, ref_answer)
    llm_eval = evaluate_llm_responses(question, model_answer, ref_answer)

    # Append results to the list
    results_list.append({
        'doc_name': doc_name,
        'question': question,
        'ref_answer': ref_answer,
        'model_answer': model_answer,
        'cosine_similarity': cosine_similarity_score,
        'bert_score': bert_score,
        'llm_eval': llm_eval
    })

# Convert the list of dictionaries to a DataFrame
results_df = pd.DataFrame(results_list)

# Save results to CSV
results_df.to_csv('results.csv', index=False)
    
    

Downloaded PDF from: https://investors.3m.com/financials/sec-filings/content/0001558370-19-000470/0001558370-19-000470.pdf to pdf_documents\3M_2018_10K.pdf


Insert of existing embedding ID: id_0
Add of existing embedding ID: id_0
Insert of existing embedding ID: id_1
Add of existing embedding ID: id_1
Insert of existing embedding ID: id_2
Add of existing embedding ID: id_2
Insert of existing embedding ID: id_3
Add of existing embedding ID: id_3
Insert of existing embedding ID: id_4
Add of existing embedding ID: id_4
Insert of existing embedding ID: id_5
Add of existing embedding ID: id_5
Insert of existing embedding ID: id_6
Add of existing embedding ID: id_6
Insert of existing embedding ID: id_7
Add of existing embedding ID: id_7
Insert of existing embedding ID: id_8
Add of existing embedding ID: id_8
Insert of existing embedding ID: id_9
Add of existing embedding ID: id_9
Insert of existing embedding ID: id_10
Add of existing embedding ID: id_10
Insert of existing embedding ID: id_11
Add of existing embedding ID: id_11
Insert of existing embedding ID: id_12
Add of existing embedding ID: id_12
Insert of existing embedding ID: id_13
Add of

Stored 12727 vectors in the Chroma vector database.
Querying Model now
{'ids': [['id_11244', 'id_11243'], ['id_11243', 'id_11242'], ['id_11242', 'id_11688'], ['id_11243', 'id_1109'], ['id_898', 'id_1555'], ['id_5275', 'id_5652'], ['id_5275', 'id_5652'], ['id_898', 'id_1555'], ['id_11243', 'id_1109'], ['id_11243', 'id_11242'], ['id_1054', 'id_1109'], ['id_898', 'id_1555'], ['id_11243', 'id_11244'], ['id_11242', 'id_11243'], ['id_7672', 'id_8819'], ['id_5241', 'id_5652'], ['id_5241', 'id_5652'], ['id_4345', 'id_6052'], ['id_898', 'id_1555'], ['id_11244', 'id_11243'], ['id_11242', 'id_11688'], ['id_11243', 'id_1109'], ['id_5275', 'id_5652'], ['id_11243', 'id_1109'], ['id_11242', 'id_11688'], ['id_11243', 'id_2486'], ['id_898', 'id_1555'], ['id_1054', 'id_1109'], ['id_1054', 'id_1109'], ['id_11243', 'id_1109'], ['id_1054', 'id_1109'], ['id_5241', 'id_5652'], ['id_11243', 'id_11244'], ['id_5275', 'id_5652'], ['id_11243', 'id_1109'], ['id_1054', 'id_1109'], ['id_2759', 'id_2901'], ['id_1054'

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


...

calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:00<00:00,  1.35it/s]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00,  9.85it/s]


done in 0.85 seconds, 1.17 sentences/sec
Average Correctness Score: 0.03
